In [ ]:
import pandas as pd
from rain_delay.preprocessing.flight_preprocessor import FlightPreprocessor
from rain_delay.data.flight_loader import FlightLoader
from rain_delay.data.dataset_builder import DatasetBuilder
from rain_delay.data.weather_client import WeatherClient
import rain_delay.data.weather_client as weather_module
from pathlib import Path
from rain_delay.data.weather_processor import WeatherProcessor
import importlib
import rain_delay.data.weather_processor as weather_module

In [ ]:
df = pd.read_csv(
    "../data/raw/flights/2025/VRA_2025_01.csv",
    sep=";",
    encoding="utf-8"
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.dtypes

In [ ]:
df.isna().mean().sort_values(ascending=False)

In [ ]:
for col in df.columns:
    print(f"\n--- {col} ---")
    print(df[col].head())

In [ ]:
preprocessor = FlightPreprocessor()
df_processed = preprocessor.transform(df)

In [ ]:
df_processed[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
    ]
].head(10)

In [ ]:
df_processed["departure_delay_minutes"].describe()

In [ ]:
df_processed["flight_status"].value_counts(dropna=False)

In [ ]:
df_processed["departure_delay_minutes"].isna().mean()

In [ ]:
df_processed.nlargest(
    10,
    "departure_delay_minutes"
)[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
        "flight_status",
    ]
]

In [ ]:
df_processed.nsmallest(
    10,
    "departure_delay_minutes"
)[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
        "flight_status",
    ]
]

In [ ]:
df_processed["departure_delay_minutes"].quantile(
    [0.001, 0.005, 0.01, 0.05, 0.95, 0.99, 0.995, 0.999]
)

In [ ]:
pd.crosstab(
    df_processed["flight_status"],
    df_processed["departure_delay_minutes"].isna(),
    normalize="index"
)

In [ ]:
df_processed.groupby("flight_status")[
    "departure_delay_minutes"
].describe()

In [ ]:
airports = pd.read_csv(
    "../data/raw/airports/airports.csv",
    sep=";",
    encoding="latin-1",
    skiprows=1,
)

airports.head()

In [ ]:
airports.columns.tolist()

In [ ]:
airports = airports[
    [
        "Código OACI",
        "Nome",
        "Município",
        "UF",
        "LATGEOPOINT",
        "LONGEOPOINT",
    ]
].copy()

In [ ]:
airports

In [ ]:
airports[airports["Código OACI"] == "SBGR"]

In [ ]:
airports = airports.rename(
    columns={
        "Código OACI": "airport_icao",
        "Nome": "airport_name",
        "Município": "city",
        "UF": "state",
        "LATGEOPOINT": "latitude",
        "LONGEOPOINT": "longitude",
    }
)

In [ ]:
df_brazil = df_processed.merge(
    airports,
    left_on="origin_airport",
    right_on="airport_icao",
    how="inner",
)

In [ ]:
df_brazil[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "departure_delay_minutes",
        "city",
        "state",
        "latitude",
        "longitude",
    ]
].head(10)

In [ ]:
print(f"Antes: {len(df_processed):,}")
print(f"Depois: {len(df_brazil):,}")
print(f"Aeroportos brasileiros: {df_brazil['origin_airport'].nunique()}")

In [ ]:
airport_origins = (
    df_brazil[
        [
            "origin_airport",
            "airport_name",
            "city",
            "state",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates()
    .sort_values("origin_airport")
    .reset_index(drop=True)
)

airport_origins.head()

In [ ]:
airport_origins.shape

In [ ]:
airport_origins[
    ["latitude", "longitude"]
].isna().sum()

In [ ]:
df_brazil["origin_airport"].value_counts().head(20)

In [ ]:
loader = FlightLoader(
    input_dir="../data/raw/flights"
)

df_all = loader.load_all()

In [ ]:
df_all.shape

In [ ]:
df_all.columns.tolist()

In [ ]:
preprocessor = FlightPreprocessor()

df_all_processed = preprocessor.transform(df_all)

In [ ]:
df_all_processed[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
    ]
].head()

In [ ]:
df_all_processed.shape

In [ ]:
df_all_brazil = df_all_processed.merge(
    airports,
    left_on="origin_airport",
    right_on="airport_icao",
    how="inner",
)

In [ ]:
print(f"Antes: {len(df_all_processed):,}")
print(f"Depois: {len(df_all_brazil):,}")
print(
    "Aeroportos brasileiros:",
    df_all_brazil["origin_airport"].nunique()
)

In [ ]:
airport_origins = (
    df_all_brazil[
        [
            "origin_airport",
            "airport_name",
            "city",
            "state",
            "latitude",
            "longitude",
        ]
    ]
    .drop_duplicates()
    .sort_values("origin_airport")
    .reset_index(drop=True)
)

In [ ]:
airport_origins.shape

In [ ]:
airport_origins[["latitude", "longitude"]].isna().sum()

In [ ]:
builder = DatasetBuilder(output_dir="../data/processed")

df_flights_final = builder.build_flights(
    flights=df_all_processed,
    airports=airports,
)

df_airports_final = builder.build_airports(
    flights_brazil=df_flights_final,
)

In [ ]:
df_flights_final.shape

In [ ]:
df_airports_final.shape

In [ ]:
df_flights_final["flight_number"] = (
    df_flights_final["flight_number"]
    .astype("string")
)

In [ ]:
df_flights_final["flight_number"].dtype

In [ ]:
df_flights_final.select_dtypes(include="object").columns.tolist()

In [ ]:
df_flights_final["Código DI"] = (
    df_flights_final["Código DI"]
    .astype("string")
)

In [ ]:
builder.save(
    flights=df_flights_final,
    airports=df_airports_final,
)

In [ ]:
df_flights_final["flight_number"] = (
    df_flights_final["flight_number"].astype("string")
)

df_flights_final["Código DI"] = (
    df_flights_final["Código DI"].astype("string")
)

In [ ]:
weather_client = WeatherClient(
    output_dir="../data/raw/weather"
)

In [ ]:
weather_gru = weather_client.download_airport(
    airport_icao="SBGR",
    latitude=-23.435556,
    longitude=-46.473056,
    start_date="2025-01-01",
    end_date="2025-01-03",
)

In [ ]:
weather_gru.head()

In [ ]:
weather_gru.shape

In [ ]:
df_airports_final[
    df_airports_final["origin_airport"] == "SBEG"
]

In [ ]:
manaus = df_airports_final[
    df_airports_final["origin_airport"] == "SBEG"
].iloc[0]

weather_manaus = weather_client.download_airport(
    airport_icao=manaus["origin_airport"],
    latitude=manaus["latitude"],
    longitude=manaus["longitude"],
    start_date="2025-01-01",
    end_date="2025-01-03",
)

In [ ]:
weather_manaus.head()

In [ ]:
weather_manaus.shape

In [ ]:
weather_manaus[["datetime", "airport_icao", "timezone"]].head()

In [ ]:
airports_parquet = pd.read_parquet("../data/processed/airports.parquet")

In [ ]:
airports_parquet.columns

In [ ]:
weather_files = list(
    Path("../data/raw/weather").glob("*.parquet")
)

print(f"Arquivos de clima: {len(weather_files)}")

In [ ]:
import pandas as pd

summary = []

for file in weather_files:
    df = pd.read_parquet(file)

    summary.append({
        "airport": file.stem,
        "rows": len(df),
        "start": df["datetime"].min(),
        "end": df["datetime"].max(),
    })

df_weather_check = pd.DataFrame(summary)

df_weather_check

In [ ]:
df_weather_check[["rows", "start", "end"]].describe()

In [ ]:
df_weather_check.sort_values("rows").head(10)

In [ ]:
weather_processor = WeatherProcessor(
    weather_dir="../data/raw/weather"
)

df_weather = weather_processor.load()

In [ ]:
df_weather

In [ ]:
weather_by_airport = (
    df_weather
    .groupby("airport_icao")
    .agg(
        rows=("datetime", "size"),
        start=("datetime", "min"),
        end=("datetime", "max"),
    )
    .sort_values("rows")
)

weather_by_airport

In [ ]:
weather_by_airport["rows"].describe()

In [ ]:
importlib.reload(weather_module)

WeatherProcessor = weather_module.WeatherProcessor

In [ ]:
weather_processor = WeatherProcessor(
    weather_dir="../data/raw/weather"
)

In [ ]:
weather_processor.save(
    weather=df_weather,
    output_path="../data/processed/weather.parquet",
)

In [ ]:
pd.read_parquet(
    "../data/processed/weather.parquet"
).shape